### PCA-Nets and Allen-Cahn Problem
Next, we considere the following differential equation:
\begin{align*}
    \epsilon \Delta u + (u^3-u)&= f &&\text{ in } (0,1)^2, \\
    n \cdot \nabla u &= 0 &&\text{ on } \partial (0,1)^2.
\end{align*}
We want to learn the mapping $f \mapsto u$. We use a data driven approach, where we have a dataset of pairs $(f_i, u_i)$, and we want to learn a neural network that approximates the solution operator.

We try this with the PCA-Net idea, where we first compute the PCA of the input and output data, and then learn a mapping between the PCA coefficients.

In [1]:
# This block is for GPU selection. Please execute.
# import pathlib
# import os
# user = int(str(pathlib.Path().resolve())[22:24])
# os.environ["CUDA_VISIBLE_DEVICES"]= str(user % 4)

In [2]:
import qewton
import matplotlib.pyplot as plt

save_path = "/localdata/tomfre/AllenCahn_data"
f_data = qewton.DEFAULT_DL_BACKEND.load_data(f"{save_path}/input_data.pt")
u_data = qewton.DEFAULT_DL_BACKEND.load_data(f"{save_path}/output_data.pt")

# print("Available data points:", len(input_u))
# print("Shape of data:", input_u.shape)

# # Plot one example of the data
# plot_idx = 0

# f, axarr = plt.subplots(1, 2, figsize=(7, 3))
# axarr[0].imshow(input_u[plot_idx, :, :], origin='lower', extent=[0,1,0,1])
# axarr[0].title.set_text(r"Example solution $u$")
# axarr[1].imshow(output_f[plot_idx, :, :], origin='lower', extent=[0,1,0,1])
# axarr[1].title.set_text(r"Example $f$")
# plt.tight_layout()

In [3]:
F = qewton.Variable("f", dim=1)
U = qewton.Variable("u", dim=1)

In [4]:
# Split data into training and testing:
total_data = len(u_data)
train_N = int(0.8*total_data)

# TODO: Build the training and test set, as in the other examples
train_f = f_data[:train_N]
train_u = u_data[:train_N]

test_f = f_data[train_N:]
test_u = u_data[train_N:]

# TODO: Create the datasets:
dataset = qewton.data.GridDataSet(
    data=[train_f, train_u], 
    feature_variables=[F, U],
)
test_dataset = qewton.data.GridDataSet(
    data=[test_f, test_u],
    feature_variables=[F, U],
)

In [5]:
data_loader = qewton.data.DataLoader(
    data_set=dataset,
    test_data_set=test_dataset,
    batch_size=4000,
    splitting_ratio=(1.0, 0.0, 0.0),
    shuffle_data=False,
)

#qewton.viz.Figure(data_loader.visualize(mode=qewton.optim.EvaluationPhase.TRAIN)).show()

In [6]:
model = qewton.algorithms.PCANet(
    input_variable=F,
    output_variable=U,
    pca_n_input=25,
    pca_n_output=400,
    data_source_node=data_loader,
    fcn_hidden_layers=2,
    fcn_hidden_neurons=50,
)

In [7]:
constraint = qewton.constraints.MSEConstraint()
computation_pipeline = qewton.Graph()
with computation_pipeline.tracker(0):
    f, u = data_loader()
    model_output = model(f, u)
    constraint(model_output, u)

computation_pipeline.setup()

In [ ]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.01,
    max_iterations=5000,
    lr_scheduler=qewton.optim.StepLR(step_size=1000, gamma=0.25)
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase],
    graphs=[computation_pipeline],
    training_objectives=[constraint],
    device=qewton.cuda(0),
)

trainer.run()

/home/tomfre/miniconda3/envs/PIONEER/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Optimization Phase 1: 100%|██████████| 5000/5000 [05:14<00:00, 15.91it/s, loss=0.0433]


In [9]:
layout = computation_pipeline.visualize(
    [model.output_ports[0], data_loader.get_output_port(U)],
    mode=qewton.optim.EvaluationPhase.TEST,
    device="cuda:0",
)
fig = qewton.viz.Figure(layout)
fig.show()